In [ ]:
# HM Land Registry 2025 price prediction
DATA_PATH = "pp-2025.csv"
MODEL_PATH = "artifacts/model.joblib"
METRICS_PATH = "artifacts/metrics.json"
IMPORTANCE_PATH = "artifacts/feature_importance.csv"
RANDOM_STATE = 42
COLUMNS = ["transaction_id", "price", "date", "postcode", "property_type", "old_new", "duration", "paon", "saon", "street", "locality", "town", "district", "county", "ppd_category", "record_status"]
FEATURES = ["property_type", "old_new", "duration", "district", "county", "outcode", "month", "quarter"]
OHE_FEATURES = ["property_type", "old_new", "duration", "quarter"]
ORD_FEATURES = ["district", "county", "outcode"]
NUM_FEATURES = ["month"]

In [ ]:
import json
from pathlib import Path
import joblib
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder

In [ ]:
df = pd.read_csv(DATA_PATH, header=None, names=COLUMNS, dtype=str)
df["price"] = pd.to_numeric(df["price"], errors="coerce")
print(df.shape)
print(df.head(3))

(954145, 16)
                           transaction_id   price              date  postcode  \
0  {42C129E4-C259-60A9-E063-4804A8C0C25D}  635000  2025-05-14 00:00  LN11 8GN   
1  {42C129E4-C25B-60A9-E063-4804A8C0C25D}  230000  2025-08-29 00:00  PE10 2BL   
2  {42C129E4-C25E-60A9-E063-4804A8C0C25D}  185000  2025-03-27 00:00  LN10 6RJ   

  property_type old_new duration paon saon              street locality  \
0             D       N        F    2  NaN        KENWICK VIEW      NaN   
1             S       N        F    6  NaN        CONWAY DRIVE      NaN   
2             S       N        F    6  NaN  KING EDWARD AVENUE      NaN   

           town        district        county ppd_category record_status  
0         LOUTH    EAST LINDSEY  LINCOLNSHIRE            A             A  
1        BOURNE  SOUTH KESTEVEN  LINCOLNSHIRE            A             A  
2  WOODHALL SPA    EAST LINDSEY  LINCOLNSHIRE            A             A  


In [ ]:
print(df.dtypes)
print(df.shape)
print(df.head())

transaction_id      str
price             int64
date                str
postcode            str
property_type       str
old_new             str
duration            str
paon                str
saon                str
street              str
locality            str
town                str
district            str
county              str
ppd_category        str
record_status       str
dtype: object
(954145, 16)
                           transaction_id   price              date  postcode  \
0  {42C129E4-C259-60A9-E063-4804A8C0C25D}  635000  2025-05-14 00:00  LN11 8GN   
1  {42C129E4-C25B-60A9-E063-4804A8C0C25D}  230000  2025-08-29 00:00  PE10 2BL   
2  {42C129E4-C25E-60A9-E063-4804A8C0C25D}  185000  2025-03-27 00:00  LN10 6RJ   
3  {42C129E4-C260-60A9-E063-4804A8C0C25D}   76000  2025-06-06 00:00  NG31 8SP   
4  {42C129E4-C261-60A9-E063-4804A8C0C25D}  237500  2025-09-18 00:00  PE11 4PP   

  property_type old_new duration paon saon              street   locality  \
0             D       N  

In [ ]:
print(df.isna().sum())
print(df["price"].isna().sum())

transaction_id         0
price                  0
date                   0
postcode            2339
property_type          0
old_new                0
duration               0
paon                   0
saon              840679
street             15688
locality          587665
town                   0
district               0
county                 0
ppd_category           0
record_status          0
dtype: int64
0


In [ ]:
print(df["price"].describe())
print(df["price"].quantile([0.25, 0.5, 0.75, 0.99, 0.999]))

count    9.541450e+05
mean     3.890188e+05
std      1.224986e+06
min      1.000000e+00
25%      1.860000e+05
50%      2.850000e+05
75%      4.300000e+05
max      7.930200e+08
Name: price, dtype: float64
0.250     186000.00
0.500     285000.00
0.750     430000.00
0.990    1850000.00
0.999    8343622.72
Name: price, dtype: float64


In [ ]:
print(df["property_type"].value_counts())
print(df["old_new"].value_counts())
print(df["duration"].value_counts())
print(df["ppd_category"].value_counts())

property_type
T    264892
S    264425
D    219440
F    157708
O     47680
Name: count, dtype: int64
old_new
N    898686
Y     55459
Name: count, dtype: int64
duration
F    743720
L    210425
Name: count, dtype: int64
ppd_category
A    794895
B    159250
Name: count, dtype: int64


In [ ]:
Path("artifacts").mkdir(exist_ok=True)
plt.figure()
plt.hist(np.log1p(df["price"].dropna()), bins=80)
plt.title("log1p price")
plt.savefig("artifacts/eda_price_hist.png")
plt.close()

In [ ]:
# category B = linked/additional transactions, not independent sales
df = df[df["ppd_category"] == "A"].copy()
print(df.shape)

(794895, 16)


In [ ]:
df = df[df["price"] > 0]
print(df.shape)

(794895, 16)


In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["month"] = df["date"].dt.month
df["quarter"] = df["date"].dt.quarter

In [ ]:
df["outcode"] = df["postcode"].str.split().str[0]
df = df[df["outcode"].notna()]

In [ ]:
cap = df["price"].quantile(0.999)
df = df[df["price"] <= cap]
print(cap)

3809064.000000013


In [ ]:
df = df.drop_duplicates(subset=["postcode", "date", "price", "paon", "street"])
print(df.shape)
print(df[FEATURES].isna().sum())

(792283, 19)
property_type    0
old_new          0
duration         0
district         0
county           0
outcode          0
month            0
quarter          0
dtype: int64
property_type    0
old_new          0
duration         0
district         0
county           0
outcode          0
month            0
quarter          0
dtype: int64


In [ ]:
X = df[FEATURES].copy()

In [ ]:
y = np.log1p(df["price"])

In [ ]:
# district/outcode exceed HGBR native categorical limit (255); ordinal encode location instead
preprocessor = ColumnTransformer([("ohe", OneHotEncoder(handle_unknown="ignore", sparse_output=False), OHE_FEATURES), ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1), ORD_FEATURES), ("num", "passthrough", NUM_FEATURES)])

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE)
print(X_train.shape, X_test.shape)

(633826, 8) (158457, 8)


In [ ]:
pipe = Pipeline([("prep", preprocessor), ("model", HistGradientBoostingRegressor(random_state=RANDOM_STATE))])

In [ ]:
pipe.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('prep', ...), ('model', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"feature_names_in_ feature_names_in_: ndarray of shape (`n_features_in_`,)Names of features seen during :term:`fit`. Only defined if theunderlying estimator exposes such an attribute when fit... versionadded:: 1.0","ndarray[object](8,)","['property_type','old_new','duration',...,'outcode','month','quarter']"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`. Only defined if theunderlying first estimator in `steps` exposes such an attributewhen fit... versionadded:: 0.24,int,8
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('ohe', ...), ('ord', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis con

In [ ]:
y_pred_log = pipe.predict(X_test)
y_true = np.expm1(y_test)
y_pred = np.expm1(y_pred_log)

In [ ]:
mae = mean_absolute_error(y_true, y_pred)
rmse = np.sqrt(mean_squared_error(y_true, y_pred))
r2 = r2_score(y_true, y_pred)
print(mae, rmse, r2)

101778.45231355171 196621.62192792748 0.5068361601153579


In [ ]:
sample_idx = np.random.default_rng(RANDOM_STATE).choice(len(y_true), size=min(5000, len(y_true)), replace=False)
plt.figure()
plt.scatter(y_true.iloc[sample_idx], y_pred[sample_idx], alpha=0.2, s=5)
plt.xlabel("actual")
plt.ylabel("predicted")
plt.savefig("artifacts/residuals.png")
plt.close()

In [ ]:
perm_idx = X_test.sample(5000, random_state=RANDOM_STATE).index
perm = permutation_importance(pipe, X_test.loc[perm_idx], y_test.loc[perm_idx], n_repeats=5, random_state=RANDOM_STATE)
importances = perm.importances_mean

In [ ]:
importance_df = pd.DataFrame({"feature": FEATURES, "importance": importances}).sort_values("importance", ascending=False)
print(importance_df)

         feature  importance
4         county    0.675236
0  property_type    0.415021
5        outcode    0.145564
3       district    0.094775
2       duration    0.069792
1        old_new    0.011149
7        quarter    0.002325
6          month   -0.000117


In [ ]:
top_n = len(FEATURES)
top = importance_df.head(top_n)
plt.figure(figsize=(8, 5))
plt.barh(top["feature"][::-1], top["importance"][::-1])
plt.xlabel("importance")
plt.title("top feature importances")
plt.tight_layout()
plt.savefig("artifacts/feature_importance.png")
plt.close()

In [ ]:
Path("artifacts").mkdir(exist_ok=True)
joblib.dump(pipe, MODEL_PATH)
importance_df.to_csv(IMPORTANCE_PATH, index=False)

In [ ]:
metrics = {"mae": mae, "rmse": rmse, "r2": r2, "n_train": len(X_train), "n_test": len(X_test)}
Path(METRICS_PATH).write_text(json.dumps(metrics, indent=2))
print(metrics)

{'mae': 101778.45231355171, 'rmse': np.float64(196621.62192792748), 'r2': 0.5068361601153579, 'n_train': 633826, 'n_test': 158457}


In [ ]:
sample = X_test.iloc[[0]]
pred_gbp = np.expm1(pipe.predict(sample))[0]
actual_gbp = np.expm1(y_test.iloc[0])
print(pred_gbp, actual_gbp)

231037.6323576041 170000.00000000012
